In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def generate_matplot_analysis(ft_csv, scratch_csv):
    # 1. Load and Clean Data
    df_ft = pd.read_csv(ft_csv)
    df_scr = pd.read_csv(scratch_csv)

    # CRITICAL: Clean whitespace from labels so they match correctly
    df_ft['Label'] = df_ft['Label'].str.strip()
    df_scr['Label'] = df_scr['Label'].str.strip()

    # 2. Merge Data
    merged = pd.merge(df_ft, df_scr, on='Label', suffixes=('_FT', '_Scratch'))
    
    # 3. Calculate AUROC Gain
    merged['AUROC_Gain'] = merged['AUROC_FT'] - merged['AUROC_Scratch']
    
    # Sort for the bar chart
    merged = merged.sort_values('AUROC_Gain', ascending=False)

    # Set the visual style
    sns.set_theme(style="whitegrid")
    plt.rcParams['figure.dpi'] = 100

    # --- PLOT 1: AUROC GAIN BAR CHART ---
    plt.figure(figsize=(10, 8))
    # Fix: Assign 'Label' to 'hue' to avoid the FutureWarning
    sns.barplot(
        data=merged, 
        x='AUROC_Gain', 
        y='Label', 
        hue='Label', 
        palette='RdYlGn_r', 
        legend=False
    )
    plt.axvline(0, color='black', linestyle='--', linewidth=1)
    
    # Fix: Use a raw string (r'') to avoid the \D SyntaxWarning
    plt.xlabel(r'$\Delta$ AUROC (Higher = Fine-Tuning is better)', fontsize=12)
    plt.title('Performance Gain: Fine-Tuning vs. Training from Scratch', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # --- PLOT 2: HEAD-TO-HEAD SCATTER ---
    plt.figure(figsize=(8, 4))
    sns.scatterplot(
        data=merged, 
        x='AUROC_Scratch', 
        y='AUROC_FT', 
        size='n_pos_FT', 
        hue='AUROC_Gain', 
        palette='viridis', 
        sizes=(40, 60),
        alpha=0.7
    )
    # Identity line (y=x)
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Equal Performance')
    
    plt.xlim(0, 1.05)
    plt.ylim(0, 1.05)
    plt.xlabel('Scratch Model AUROC', fontsize=12)
    plt.ylabel('Fine-Tuned Model AUROC', fontsize=12)
    plt.title('Comparison of Individual Diagnostic Classes', fontsize=14, fontweight='bold')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()

    # --- PLOT 3: PREVALENCE VS GAIN ---
    plt.figure(figsize=(8, 3))
    sns.regplot(
        data=merged, 
        x='n_pos_FT', 
        y='AUROC_Gain', 
        scatter_kws={'alpha':0.5}, 
        line_kws={'color':'red'},
        logx=True
    )
    plt.xscale('log')
    plt.xlabel('Number of Positive Samples (Log Scale)', fontsize=12)
    plt.ylabel(r'$\Delta$ AUROC Gain', fontsize=12)
    plt.title('Fine-Tuning Benefit vs. Data Scarcity', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Print a verified summary
    print("\n=== Verified Summary Statistics ===")
    print(f"Labels successfully compared: {len(merged)}")
    print(f"Mean AUROC (FT):      {merged['AUROC_FT'].mean():.4f}")
    print(f"Mean AUROC (Scratch): {merged['AUROC_Scratch'].mean():.4f}")
    print(f"Avg Improvement:      {merged['AUROC_Gain'].mean():.4f}")

# generate_matplot_analysis('ft_500_test_resutls.csv', 'medium_500_test_results.csv')
#generate_ecg_analysis('ft_500_test_resutls.csv', 'medium_500_test_results.csv')
generate_matplot_analysis( 'res/ablation_ft/ft_500/ft_500_test_results.csv','res/ablation_resnet/medium_500/medium_500_test_results.csv')

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

def generate_ecg_analysis(ft_csv, scratch_csv):
    # 1. Load Data
    df_ft = pd.read_csv(ft_csv)
    df_scr = pd.read_csv(scratch_csv)

    # 2. Clean Labels
    df_ft['Label'] = df_ft['Label'].str.strip()
    df_scr['Label'] = df_scr['Label'].str.strip()

    # 3. Merge Data
    merged = pd.merge(df_ft, df_scr, on='Label', suffixes=('_FT', '_Scratch'))
    
    # 4. Calculate Metrics
    merged['AUROC_Gain'] = merged['AUROC_FT'] - merged['AUROC_Scratch']
    
    # 5. Print Summary
    print(f"=== Verified Summary ===")
    print(f"Total Labels matched: {len(merged)}")
    print(f"Mean FT AUROC: {merged['AUROC_FT'].mean():.4f}")
    print(f"Mean Scratch AUROC: {merged['AUROC_Scratch'].mean():.4f}")
    print(f"Average Gain: {merged['AUROC_Gain'].mean():.4f}")

    # --- PLOT 1: Horizontal Bar Chart ---
    fig1 = px.bar(
        merged.sort_values('AUROC_Gain'),
        x='AUROC_Gain',
        y='Label',
        orientation='h',
        color='AUROC_Gain',
        color_continuous_scale='RdYlGn',
        title='AUROC Improvement: Fine-Tuned vs. Scratch',
        labels={'AUROC_Gain': 'Delta AUROC'},
        template='plotly_white'
    )
    fig1.add_vline(x=0, line_dash="dash", line_color="black")
    # Reduced size
    fig1.update_layout(height=600, width=800, coloraxis_showscale=False)
    fig1.show()

    # --- PLOT 2: Head-to-Head Scatter (Cleaned Blobs) ---
    fig2 = px.scatter(
        merged,
        x='AUROC_Scratch',
        y='AUROC_FT',
        size='n_pos_FT',
        color='AUROC_Gain',
        hover_name='Label',
        # Removed text='Label' to clean up the blobs
        title='Model Comparison: FT vs. Scratch Performance',
        labels={'AUROC_Scratch': 'Scratch AUROC', 'AUROC_FT': 'Fine-Tuned AUROC'},
        template='plotly_white'
    )
    fig2.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(color="Red", dash="dot"))
    # Reduced size
    fig2.update_layout(
        height=500, 
        width=600, 
        xaxis=dict(range=[0, 1.05]), 
        yaxis=dict(range=[0, 1.05])
    )
    fig2.show()

    # --- PLOT 3: Prevalence vs Gain ---
    fig3 = px.scatter(
        merged,
        x='n_pos_FT',
        y='AUROC_Gain',
        log_x=True,
        trendline="ols",
        hover_name='Label',
        title='Fine-Tuning Benefit vs. Class Prevalence',
        labels={'n_pos_FT': 'Samples (Log)', 'AUROC_Gain': 'AUROC Gain'},
        template='plotly_white'
    )
    # Reduced size
    fig3.update_layout(height=500, width=600)
    fig3.show()

# Run the analysis
generate_ecg_analysis('res/ablation_ft/ft_500/ft_500_test_results.csv', 'res/ablation_resnet/small_500/small_500_test_results.csv')

=== Verified Summary ===
Total Labels matched: 38
Mean FT AUROC: 0.7641
Mean Scratch AUROC: 0.6226
Average Gain: 0.1415


In [2]:

# Run the analysis
generate_ecg_analysis('res/ablation_net1d/small_500/small_500_test_results.csv', 'res/ablation_resnet/small_500/small_500_test_results.csv')

=== Verified Summary ===
Total Labels matched: 38
Mean FT AUROC: 0.7036
Mean Scratch AUROC: 0.6226
Average Gain: 0.0810


In [1]:
import pandas as pd

In [3]:
pd.read_csv('C:/Users/zoorab/Desktop/zoher/University/Projects/ECGFounder/ecg_with_exact_match.csv')

,Filename,ECG_ID,Patient_ID,Age,Gender,Acquisition_date,Sampling_point,Lead,AHA_code,CHN_code,...,basSQI,bSQI,Disease_desc,label,matched_types,dominant_match_type,label_count,num_labels,Age_days,Age_years
0,P00/P00002/P00002_E01,P00002_E01,P00002,4327d,'Male',2017-11-28 21:59:47,15000,12,'C21','C13',...,'I':0.995;'II':0.980;'III':0.992;'aVR':0.992;'...,'I':1.000;'II':1.000;'III':1.000;'aVR':1.000;'...,Sinus tachycardia,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",['Exact'],Exact,1,1,4327,11.854795
1,P00/P00003/P00003_E01,P00003_E01,P00003,1087d,'Female',2017-11-29 16:04:57,10000,12,'C21','C13',...,'I':0.915;'II':0.895;'III':0.882;'aVR':0.908;'...,'I':1.000;'II':1.000;'III':1.000;'aVR':1.000;'...,Sinus tachycardia,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",['Exact'],Exact,1,1,1087,2.978082
2,P00/P00005/P00005_E01,P00005_E01,P00005,857d,'Male',2017-12-09 09:58:20,15000,12,'C21';'First-degree AV block';'L147','C13';'H74';'L123',...,'I':0.971;'II':0.964;'III':0.992;'aVR':0.966;'...,'I':1.000;'II':1.000;'III':1.000;'aVR':1.000;'...,Sinus tachycardia; First-degree AV block; T-wa...,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, ...","['Exact', 'Exact', 'Exact']",Exact,3,3,857,2.347945
3,P00/P00006/P00006_E01,P00006_E01,P00006,889d,'Female',2017-12-10 07:38:42,15000,12,'D37';'H83';'I105';'L155','D28';'H75';'I86';'L133',...,'I':0.996;'II':0.990;'III':0.995;'aVR':0.994;'...,'I':1.000;'II':1.000;'III':1.000;'aVR':1.000;'...,Junctional escape complex(es); Second-degree A...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","['Exact', 'Exact', 'Exact']",Exact,3,4,889,2.435616
4,P00/P00006/P00006_E02,P00006_E02,P00006,889d,'Female',2017-12-10 00:41:53,15000,12,'D37';'H84';'Left ventricular high voltage';'L...,'D28';'H76';'J106';'L133',...,'I':0.992;'II':0.987;'III':0.992;'aVR':0.990;'...,'I':1.000;'II':1.000;'III':1.000;'aVR':1.000;'...,Junctional escape complex(es); Second-degree A...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","['Exact', 'Exact']",Exact,2,4,889,2.435616
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9392,P11/P11635/P11635_E01,P11635_E01,P11635,2904d,'Female',2021-06-28 09:24:37,15000,12,'L147';'Suggests208','L123';'Hyperkalemia',...,'I':0.909;'II':0.963;'III':0.937;'aVR':0.956;'...,'I':0.993;'II':0.986;'III':1.000;'aVR':0.993;'...,T-wave abnormality; Hyperkalemia,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...",['Exact'],Exact,1,2,2904,7.956164
9393,P11/P11637/P11637_E01,P11637_E01,P11637,3901d,'Female',2021-06-24 09:44:42,10000,12,'L147','L123',...,'I':0.992;'II':0.995;'III':0.993;'aVR':0.994;'...,'I':0.971;'II':0.989;'III':1.000;'aVR':0.980;'...,T-wave abnormality,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...",['Exact'],Exact,1,1,3901,10.687671
9394,P11/P11638/P11638_E01,P11638_E01,P11638,5147d,'Male',2021-07-01 09:46:08,15000,12,'C22';'C23','C14';'C15',...,'I':0.964;'II':0.974;'III':0.982;'aVR':0.971;'...,'I':1.000;'II':0.986;'III':1.000;'aVR':1.000;'...,Sinus bradycardia; Sinus arrhythmia,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",['Exact'],Exact,1,2,5147,14.101370
9395,P11/P11639/P11639_E01,P11639_E01,P11639,2646d,'Male',2021-06-24 18:22:31,10000,12,'A1','A1',...,'I':0.991;'II':0.991;'III':0.981;'aVR':0.992;'...,'I':1.000;'II':1.000;'III':0.990;'aVR':1.000;'...,Normal ECG,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",['Exact'],Exact,1,1,2646,7.249315


In [4]:
import pandas as pd

# Load the original data
df = pd.read_csv('ecg_with_exact_match.csv')

# Mapping of your requested names to the strings found in the CSV
target_map = {
    'Atrial Fibrillation': 'atrial fibrillation',
    'Atrial Flutter': 'atrial flutter',
    'Incomplete Right Bundle Branch Block': 'incomplete right bundle-branch block',
    'Left Anterior Fascicular Block': 'left anterior fascicular block',
    'Left Bundle Branch Block': 'left bundle-branch block',
    'Left Axis Deviation': 'left-axis deviation',
    'Right Axis Deviation': 'right-axis deviation',
    'Right Bundle Branch Block': 'right bundle-branch block',
    'Sinus Bradycardia': 'sinus bradycardia',
    'Sinus Tachycardia': 'sinus tachycardia'
}

def extract_samples(df, mapping, n=5):
    all_samples = []
    
    # Process descriptions into lists for accurate matching
    # (prevents matching "Right Bundle" inside "Incomplete Right Bundle")
    df['disease_list'] = df['Disease_desc'].fillna('').str.lower().str.split('; ')
    
    for category, search_term in mapping.items():
        # Filter for rows containing the specific disease segment
        mask = df['disease_list'].apply(lambda x: search_term in x)
        available = df[mask]
        
        if len(available) >= n:
            sampled = available.sample(n=n, random_state=42)
        else:
            sampled = available # Take all if fewer than 5 exist
            
        sampled['Target_Category'] = category
        all_samples.append(sampled)
        
    # Combine results and clean up
    result = pd.concat(all_samples).drop(columns=['disease_list'])
    return result

# Execute and save
extracted_df = extract_samples(df, target_map)
extracted_df.to_csv('extracted_ecg_samples.csv', index=False)

print(f"Extraction complete. {len(extracted_df)} samples saved.")

Extraction complete. 50 samples saved.
